In [1]:
import requests
import urllib
import re
import unicodedata
from bs4 import BeautifulSoup
import aiohttp
import asyncio

def remove_accents(input_str):
    """Removes accents from a string using Unicode normalization."""
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

def normalize_entry(line: str) -> str:
    """
    Normaliza o texto dividindo as linhas em listas separatdas
    """
    line = line.replace("\n", "")
    line = re.sub(r'[^\w\s]', '', line)  # Remove punctuation
    line = re.sub(r'\s+', ' ', line).strip()  # Remove extra whitespace
    line = line.replace(" ", "-")
    line = line.lower()  # Convert to lowercase
    line = remove_accents(line)

    return line

def list_text(line: str) -> str:
    """
    Normaliza o texto dividindo as linhas em listas separatdas
    """
    line = re.sub(r'[^\w\s]', '', line)  # Remove punctuation
    line = re.sub(r'\s+', ' ', line).strip()  # Remove extra whitespace
    line = line.lower()  # Convert to lowercase
    line = remove_accents(line)

    return line



In [3]:
async def fetch_url(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            return await response.text(), response.status, response.url

musica = "A Felicidade"
artista = "Tom Zé"
key = 1
query_letra = urllib.parse.quote_plus(f"{musica} {artista} letra")
url_cifraclub = "https://www.cifraclub.com.br/" + normalize_entry(artista) + "/" + normalize_entry(musica) + "/#key=" + str(key)
print(url_cifraclub)

result, status, real_url= await fetch_url(url_cifraclub)

print(real_url)
print(status)

dom = BeautifulSoup(result, "html.parser")
body = dom.body
pre = body.find("pre")

https://www.cifraclub.com.br/tom-ze/a-felicidade/#key=1
https://www.cifraclub.com.br/tom-ze/a-felicidade/
403


In [219]:
lines = pre.get_text().split('\n')
lines

['[Intro] Em/B  Em/D  Em',
 '        Em/G  Em/F#  Em',
 '        Em/B  Em/D  Em',
 '        Em/G  Em/F#  Em',
 '',
 '[Tab Riff - Intro] ',
 '',
 'Parte 1 de 3',
 '   Em/B  Em/D    Em      Em/G    Em/F#       Em',
 'E|-------------0-------0-------------------------|',
 'B|-----0---0---------0-0-----------------------0-|',
 'G|---0-------0-----0---0-----------------------0-|',
 'D|-------0-------2-----------------------------2-|',
 'A|-2---------------------0h2\\1p0---1/2\\1p0-----2-|',
 'E|-------------------------------3---------3-0---|',
 '',
 'Parte 2 de 3',
 '   Em/B  Em/D    Em       ',
 'E|-------------0-------0-------------------------|',
 'B|-----0---0---------0-0-------------------------|',
 'G|---0-------0-----0---0-------------------------|',
 'D|-------0-------2-------------------------------|',
 'A|-2---------------------------------------------|',
 'E|-----------------------------------------------|',
 '',
 'Parte 3 de 3',
 '   Em/G   Em/F# Em',
 'E|-----------------------

In [220]:
def normalizer(lines):
    #eliminate chord lines
    #chord_regex = r"^[A-G][ABCDEFGabcdefgm#°+-/()M123456789]"
    chord_regex = r"^[A-G]"
    new_lines = []
    for line in lines:
        cs = 0
        notcs = 0
        th = 0.8
        for word in line.replace("/", " ").replace("(", "").replace(")", "").split():
            if re.match(chord_regex, word):
                cs += 1
            else:
                notcs += 1
        if cs + notcs == 0:
            continue
        if cs / (cs + notcs) < th:
            new_lines.append(line.replace("\n", ""))

    lines = new_lines

    #eliminate between square brackets []
    new_lines = []
    for line in lines:
        if "[" in line and "]" in line:
            continue
        new_lines.append(line)

    lines = new_lines

    # eliminate tab lines (many - char)
    new_lines = []
    for line in lines:
        if line.count("-") > 5:
            continue
        new_lines.append(line)

    lines = new_lines

    new_lines = []
    for line in lines:
        new_lines.append(re.sub(r"\s+", " ", line).strip().lower())

    return new_lines

In [221]:
norm_lines = normalizer(lines)

In [188]:
text_to_index = str(pre).replace("<pre>", "").replace("</pre>", "").split('\n')

In [222]:
def scrolling_index(norm_lines, text_to_index, **kwargs):
    indexed_text = {}
    if kwargs:
        indexed_text['title'] = kwargs['title']
        indexed_text['author'] = kwargs['author']
        i=0
    else:
        indexed_text['title'] = text[0]
        indexed_text['author'] = text[1]
        i=3

    j = 0
    text_content  = []
    for i, norm_line in enumerate(norm_lines):
        tti = text_to_index[j:]
        for line in tti:
            if norm_line in re.sub(r"\s+", " ", line).strip().lower():
                text_content.append(( 'l' + str(i) , line.replace("\n", "")))
                j += 1
                break
            else:
                text_content.append(('<BR>', "<pre>" + line.replace("\n", "") + "</pre>"))
                j += 1
            

    indexed_text['content'] = text_content
    indexed_text['content_length'] = len(text_content)
    indexed_text['index_count'] = len(norm_lines)

    return indexed_text


In [223]:
tendexed_text = scrolling_index(norm_lines, text_to_index, title='Construção', author='Chico Buarque')

In [224]:
tendexed_text

{'title': 'Construção',
 'author': 'Chico Buarque',
 'content': [('<BR>',
   '<pre>[Intro] <b>Em/B</b>  <b>Em/D</b>  <b>Em</b></pre>'),
  ('<BR>', '<pre>        <b>Em/G</b>  <b>Em/F#</b>  <b>Em</b></pre>'),
  ('<BR>', '<pre>        <b>Em/B</b>  <b>Em/D</b>  <b>Em</b></pre>'),
  ('<BR>', '<pre>        <b>Em/G</b>  <b>Em/F#</b>  <b>Em</b></pre>'),
  ('<BR>', '<pre></pre>'),
  ('<BR>', '<pre>[Tab Riff - Intro] </pre>'),
  ('<BR>', '<pre></pre>'),
  ('l0', '<span class="tablatura">Parte 1 de 3'),
  ('<BR>',
   '<pre>   <b>Em/B</b>  <b>Em/D</b>    <b>Em</b>      <b>Em/G</b>    <b>Em/F#</b>       <b>Em</b></pre>'),
  ('<BR>',
   '<pre><span class="cnt">E|-------------0-------0-------------------------|</pre>'),
  ('<BR>', '<pre>B|-----0---0---------0-0-----------------------0-|</pre>'),
  ('<BR>', '<pre>G|---0-------0-----0---0-----------------------0-|</pre>'),
  ('<BR>', '<pre>D|-------0-------2-----------------------------2-|</pre>'),
  ('<BR>', '<pre>A|-2---------------------0h2\\1p0---1

In [206]:
norm_lines[6]

'e atravessou a rua com seu passo tímido'

In [212]:
text_to_index[43]

'E atravessou a rua com seu passo  tímido'

In [213]:
norm_lines[6] in text_to_index[43].lower()

False